This notebook creates fixed sized portfolios - (approach changed based on initial testing results)

In [1]:
import pandas as pd
import numpy as np
import time
import datetime
import os
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler


In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [3]:
port_size = 20
user_id_item_size = 10

# Utils

In [4]:
def split_symbol(symbol):
    return symbol.split('.')[0]

In [5]:
def mapper(symb, mapper):
    try:
        return mapper.get(symb)
    except:
        return 'Empty'

In [6]:
def get_max_values(df, col_name = 'RATING', round_method = 'round'):
    max_rating_row = df.loc[df[col_name].idxmax()]
    if round_method == 'round':
        max_rating_row[col_name] = max_rating_row[col_name].round(0)
        
    elif round_method == 'ceil':
        max_rating_row[col_name] = np.ceil(max_rating_row[col_name].array)
    
    return max_rating_row

In [7]:
def infer_rating(df, qnt_col = 'SHARESQTY', price_col= 'SHAREPRICE', rating_col = 'RATING'):

    order_prices = df[qnt_col] * df[price_col]
    scaler = MinMaxScaler(feature_range=(1,5))
    scaled_price = scaler.fit_transform(order_prices.values.reshape(-1, 1))
    df[rating_col] = np.clip(scaled_price, 1, 5)

    df = df.groupby("STOCKCODE", group_keys = False).apply(lambda x: get_max_values(x)).reset_index(drop = True)
    return df

In [8]:
def to_timestamp(date):
    return datetime.datetime.timestamp(date)

In [9]:
def create_user_portfolios(df, time_col = 'UNIX_TS', latest_k = port_size, user_id_item_size = user_id_item_size):
    df = df.sort_values(time_col, ascending = False)
    df = df.tail(latest_k)
    user_id_as_items = df.iloc[:user_id_item_size].STOCKCODE.values
    df['USER_ID'] = (np.repeat(user_id_as_items.reshape(1,-1), len(df), axis = 0)).tolist()

    return df


# Data reading


In [10]:
data_path = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2\raw"

portfolios = pd.DataFrame()
for fname in os.listdir(data_path):
    
    broker_df_ = pd.read_csv(os.path.join(data_path,fname), sep = '|')
    print("--- reading : {}".format(fname))
    
    portfolios = pd.concat([portfolios, broker_df_], ignore_index = True)

C:\Users\naradaw\AppData\Local\Temp\ipykernel_37600\480125734.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  broker_df_ = pd.read_csv(os.path.join(data_path,fname), sep = '|')


--- reading : Bartleet.txt


C:\Users\naradaw\AppData\Local\Temp\ipykernel_37600\480125734.py:6: DtypeWarning: Columns (2,7) have mixed types. Specify dtype option on import or set low_memory=False.
  broker_df_ = pd.read_csv(os.path.join(data_path,fname), sep = '|')


--- reading : CAS.txt
--- reading : FCE.txt
--- reading : NLE.txt
--- reading : RPS.txt


In [11]:
portfolios['TRADE_TIME'] = pd.to_datetime(portfolios['TRADE_TIME'])
portfolios['TRADE_DATE'] = pd.to_datetime(portfolios['TRADE_DATE'])

In [12]:
stock_info = pd.read_excel('../../data/stock_data.xlsx')
stock_info = stock_info.drop(['Unnamed: 0'],axis = 1)
stock_info.shape

(282, 4)

# Preprocessing

In [13]:
portfolios.shape

(3728894, 8)

In [14]:
portfolios.head(2)

,CDSACCNO,STOCKCODE,REFERANCE,TRAN_TYPE,SHARESQTY,SHAREPRICE,TRADE_DATE,TRADE_TIME
0,BMS-731900310-VN/00,AGPL.N0000,2024153263,B,125,7.5,2024-05-13,2024-05-13 12:25:45
1,BMS-800262640-VN/00,RIL.N0000,2024153264,S,-100,8.5,2024-05-13,2024-05-13 12:25:48


In [15]:
portfolios.dtypes

CDSACCNO              object
STOCKCODE             object
REFERANCE             object
TRAN_TYPE             object
SHARESQTY              int64
SHAREPRICE           float64
TRADE_DATE    datetime64[ns]
TRADE_TIME    datetime64[ns]
dtype: object

In [16]:
stock_info = stock_info.dropna()

In [17]:
# Only selecting Buy orders
portfolios_df = portfolios.copy()
portfolios_df = portfolios_df.loc[portfolios_df.TRAN_TYPE == 'B']
portfolios_df.shape

(1869633, 8)

In [18]:
portfolios_df['STOCKCODE'] = portfolios_df.STOCKCODE.apply(lambda x : split_symbol(x))

In [19]:
#remove symbols that do not have their details in the stock data dataset
unique_port_symbols = set(portfolios_df.STOCKCODE.unique())
unique_symbols = set(stock_info.symbol.unique())

print("unique symbols in portfolio data : {} | unique_symbols in stock details : {}".format(len(unique_port_symbols), len(unique_symbols)))
to_remove = list(unique_port_symbols - unique_symbols)
print("removing {} symbols: {}".format(len(to_remove),to_remove))
portfolios_df = portfolios_df[~portfolios_df.STOCKCODE.isin(to_remove)]

unique symbols in portfolio data : 288 | unique_symbols in stock details : 280
removing 13 symbols: ['CITW', 'CBNK', 'CLC', 'WIND', 'YORK', 'CALI', 'LGIL', 'GSF', 'WATA', 'AGPL', 'PDL', 'SFL', 'UBF']


In [20]:
# dropping users with less than k unique items in their portfolio

portfolios_df_fil_1 = portfolios_df.groupby(by = 'CDSACCNO').filter(lambda x: x['STOCKCODE'].nunique() > port_size)
portfolios_df_fil_1['UNIX_TS'] = portfolios_df_fil_1['TRADE_DATE'].apply(lambda x: to_timestamp(x))
portfolios_df_fil_1.head()

,CDSACCNO,STOCKCODE,REFERANCE,TRAN_TYPE,SHARESQTY,SHAREPRICE,TRADE_DATE,TRADE_TIME,UNIX_TS
3,BMS-42281-LI/00,PACK,2024153266,B,300,15.00,2024-05-13,2024-05-13 12:26:03,1.715539e+09
4,BMS-478-LC/00,HNB,2024153267,B,350,202.25,2024-05-13,2024-05-13 12:26:42,1.715539e+09
11,BMS-478-LC/00,HNB,2024153274,B,109,202.25,2024-05-13,2024-05-13 12:29:02,1.715539e+09
13,BMS-522272426-VN/00,ACAP,2024153276,B,350,3.80,2024-05-13,2024-05-13 12:29:24,1.715539e+09
19,BMS-731142378-VN/00,VPEL,2024153282,B,2627,7.90,2024-05-13,2024-05-13 12:30:07,1.715539e+09


In [21]:
portfolios_df_fil_1.CDSACCNO.nunique(), portfolios_df_fil_1.STOCKCODE.nunique()

print("print number of unique users : {}".format(portfolios_df_fil_1.CDSACCNO.nunique()))
print("print number of unique items : {}".format(portfolios_df_fil_1.STOCKCODE.nunique()))

print number of unique users : 2796
print number of unique items : 275


In [22]:
symb_to_name = dict(zip(stock_info.symbol,stock_info.name))
symb_to_gics = dict(zip(stock_info.symbol, stock_info.gics_code))

portfolios_df_fil_1['STOCKNAME'] = portfolios_df_fil_1.STOCKCODE.apply(lambda x: mapper(x,symb_to_name))
portfolios_df_fil_1['GICS'] = portfolios_df_fil_1.STOCKCODE.apply(lambda x: mapper(x,symb_to_gics))

In [25]:
"""
1. calculate ratings for each user and gets max rating for each symbol -> a symbol can only apear once in 
   a user portfolio and the value that appears is the max infered rating for that symbol

2. takes the latest k number of symbols that the user has interacted with -> each user can only have k number of items in the portfolio

3. creates a USER_ID with the latest m number of items user has bought.
"""

portfolios_df_fil_3 = portfolios_df_fil_1.groupby('CDSACCNO', group_keys= False).apply(lambda x: infer_rating(x)).groupby('CDSACCNO', group_keys= False).apply(lambda x: create_user_portfolios(x, latest_k= port_size)).reset_index(drop = True) #.sort_values('RATING', ascending= False)

In [26]:
portfolios_df_fil_3.head(2)

,CDSACCNO,STOCKCODE,REFERANCE,TRAN_TYPE,SHARESQTY,SHAREPRICE,TRADE_DATE,TRADE_TIME,UNIX_TS,STOCKNAME,GICS,RATING,USER_ID
0,BMS-10544-LC/00,SLTL,2023329419,B,75,102.0,2023-07-18,2023-07-18 10:22:31,1.689619e+09,SRI LANKA TELECOM PLC,Telecommunication Services,1.0,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ..."
1,BMS-10544-LC/00,RIL,197511,B,2500,5.6,2023-04-12,NaT,1.681238e+09,R I L PROPERTY PLC,Retailing,1.0,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ..."


In [27]:
portfolios_df_fil_3.shape

(55920, 13)

In [28]:
portfolios_df_fil_3.CDSACCNO.value_counts()

CDSACCNO
BMS-10544-LC/00        20
COM-42260-LC/00        20
COM-32508-LI/00        20
COM-33565-LI/00        20
COM-34293-LI/00        20
                       ..
BMS-73603-LI/00        20
BMS-736471787-VN/00    20
BMS-736711710-VN/00    20
BMS-73817-LC/00        20
SCB-1522-LC/00         20
Name: count, Length: 2796, dtype: int64

In [29]:
stockcode_srs = portfolios_df_fil_3.STOCKCODE.value_counts()
stockcode_srs = stockcode_srs[stockcode_srs<10]
symbols_to_remove = list(stockcode_srs.index)

In [30]:
portfolios_df_fil_3 = portfolios_df_fil_3[~portfolios_df_fil_3.STOCKCODE.isin(symbols_to_remove)]
portfolios_df_fil_3.CDSACCNO.value_counts()

CDSACCNO
BMS-10544-LC/00        20
COM-42260-LC/00        20
COM-32508-LI/00        20
COM-33565-LI/00        20
COM-34293-LI/00        20
                       ..
BMS-79100-LI/00        19
BMS-531080360-VN/00    19
CAS-821813450-VN/00    18
BMS-820884310-VN/00    18
HDF-741621630-VN/00    18
Name: count, Length: 2796, dtype: int64

In [31]:
portfolios_df_fil_3.CDSACCNO.nunique(), portfolios_df_fil_3.STOCKCODE.nunique()

(2796, 263)

In [32]:
portfolios_df_fil_3.head(2)

,CDSACCNO,STOCKCODE,REFERANCE,TRAN_TYPE,SHARESQTY,SHAREPRICE,TRADE_DATE,TRADE_TIME,UNIX_TS,STOCKNAME,GICS,RATING,USER_ID
0,BMS-10544-LC/00,SLTL,2023329419,B,75,102.0,2023-07-18,2023-07-18 10:22:31,1.689619e+09,SRI LANKA TELECOM PLC,Telecommunication Services,1.0,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ..."
1,BMS-10544-LC/00,RIL,197511,B,2500,5.6,2023-04-12,NaT,1.681238e+09,R I L PROPERTY PLC,Retailing,1.0,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ..."


In [33]:
portfolios_df_fil_3.GICS.nunique()

33

In [34]:
portfolios_df_fil_4 = portfolios_df_fil_3[['USER_ID','CDSACCNO','STOCKCODE','UNIX_TS','RATING','GICS','STOCKNAME']] #,'GICS','STOCKNAME'

In [35]:
portfolios_df_fil_4

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
0,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,SLTL,1.689619e+09,1.0,Telecommunication Services,SRI LANKA TELECOM PLC
1,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,RIL,1.681238e+09,1.0,Retailing,R I L PROPERTY PLC
2,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,EML,1.678991e+09,1.0,Commercial & Professional Services,E M L CONSULTANTS PLC
3,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,NDB,1.678732e+09,1.0,Banks,NATIONAL DEVELOPMENT BANK PLC
4,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,SAMP,1.678300e+09,1.0,Banks,SAMPATH BANK PLC
...,...,...,...,...,...,...,...
55915,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,CIC,1.667500e+09,1.0,Materials,C I C HOLDINGS PLC
55916,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,HHL,1.667155e+09,2.0,Capital Goods,HEMAS HOLDINGS PLC
55917,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,KHL,1.662662e+09,1.0,Consumer Services,JOHN KEELLS HOTELS PLC
55918,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,AHUN,1.655318e+09,1.0,Consumer Services,AITKEN SPENCE HOTEL HOLDINGS PLC


In [36]:
train_ = portfolios_df_fil_4.groupby('CDSACCNO', group_keys=False).apply(lambda x: x.sample(frac=0.8))
train_

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
1,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,RIL,1.681238e+09,1.0,Retailing,R I L PROPERTY PLC
10,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,EXPO,1.652035e+09,2.0,Transpotation,EXPOLANKA HOLDINGS PLC
5,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,EMER,1.662057e+09,1.0,Retailing,EASTERN MERCHANTS PLC
13,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,SUN,1.646764e+09,1.0,Food Beverage & Tobacco,SUNSHINE HOLDINGS PLC
2,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,EML,1.678991e+09,1.0,Commercial & Professional Services,E M L CONSULTANTS PLC
...,...,...,...,...,...,...,...
55911,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,JKH,1.682534e+09,2.0,Capital Goods,JOHN KEELLS HOLDINGS PLC
55902,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,MGT,1.691606e+09,2.0,Consumer Durables & Apparel,HAYLEYS FABRIC PLC
55900,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,COMB,1.693852e+09,2.0,Banks,COMMERCIAL BANK OF CEYLON PLC
55908,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,AEL,1.685644e+09,1.0,Capital Goods,ACCESS ENGINEERING PLC


In [37]:
test_ = portfolios_df_fil_4[~portfolios_df_fil_4.index.isin(train_.index)]
test_

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
0,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,SLTL,1.689619e+09,1.0,Telecommunication Services,SRI LANKA TELECOM PLC
3,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,NDB,1.678732e+09,1.0,Banks,NATIONAL DEVELOPMENT BANK PLC
6,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,COCO,1.661366e+09,1.0,Food Beverage & Tobacco,RENUKA FOODS PLC
19,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,VONE,1.641148e+09,3.0,Utilities,VALLIBEL ONE PLC
25,"[SUN, SAMP, RIL, VONE, COMB, DFCC, LMF, TAP, G...",BMS-11214-LC/00,DFCC,1.641753e+09,2.0,Banks,DFCC BANK PLC
...,...,...,...,...,...,...,...
55898,"[LLUB, GLAS, DIST, COMB, MGT, PLC, ALUM, SUN, ...",SCB-11577-LC/00,AHUN,1.655318e+09,1.0,Consumer Services,AITKEN SPENCE HOTEL HOLDINGS PLC
55903,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,ALUM,1.691087e+09,1.0,Materials,ALUMEX PLC
55904,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,SUN,1.689014e+09,5.0,Food Beverage & Tobacco,SUNSHINE HOLDINGS PLC
55907,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,JAT,1.687372e+09,2.0,Materials (1510),JAT HOLDINGS PLC


In [38]:
portfolios_df_fil_4.shape[0] == train_.shape[0] + test_.shape[0]

True

In [39]:
train_.CDSACCNO.nunique(), test_.CDSACCNO.nunique()

(2796, 2796)

In [40]:
train_.STOCKCODE.nunique(), test_.STOCKCODE.nunique()

(263, 263)

In [58]:
len(train_), len(test_)

(44688, 11184)

In [41]:
test_.CDSACCNO.value_counts()

CDSACCNO
BMS-10544-LC/00        4
COM-42260-LC/00        4
COM-32508-LI/00        4
COM-33565-LI/00        4
COM-34293-LI/00        4
                      ..
BMS-73603-LI/00        4
BMS-736471787-VN/00    4
BMS-736711710-VN/00    4
BMS-73817-LC/00        4
SCB-1522-LC/00         4
Name: count, Length: 2796, dtype: int64

In [42]:
import sys

sys.exit()

SystemExit: 

c:\Users\bpadmin\anaconda3\envs\atrad_cars_v2\lib\site-packages\IPython\core\interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [43]:
data_dict = portfolios_df_fil_4.to_dict(orient='list')
dataset = tf.data.Dataset.from_tensor_slices(data_dict)
dataset.save("../../data/portfolios_v2_fixed_port_size_{}_useridseq/portfolios".format(port_size))

In [44]:
data_dict = train_.to_dict(orient='list')
train_dataset = tf.data.Dataset.from_tensor_slices(data_dict)
train_dataset.save("../../data/portfolios_v2_fixed_port_size_{}_useridseq/retriver_train".format(port_size))

In [59]:
len(train_dataset)

44688

In [45]:
data_dict = test_.to_dict(orient='list')
test_dataset = tf.data.Dataset.from_tensor_slices(data_dict)
test_dataset.save("../../data/portfolios_v2_fixed_port_size_{}_useridseq/retriver_test".format(port_size))

In [46]:
len(dataset), len(train_dataset), len(test_dataset)

(55872, 44688, 11184)

In [47]:
train_hoo, test_hoo = pd.DataFrame(), pd.DataFrame()
for name, group in portfolios_df_fil_4.groupby('CDSACCNO'):
  train_hoo = pd.concat([train_hoo, group.iloc[0:-1]], ignore_index=True)  # Take first row as test
  test_hoo = pd.concat([test_hoo, group.iloc[-1:]], ignore_index=True)  # Rest for train

In [48]:
train_hoo.CDSACCNO.nunique(), test_hoo.CDSACCNO.nunique()

(2796, 2796)

In [49]:
train_hoo

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
0,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,SLTL,1.689619e+09,1.0,Telecommunication Services,SRI LANKA TELECOM PLC
1,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,RIL,1.681238e+09,1.0,Retailing,R I L PROPERTY PLC
2,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,EML,1.678991e+09,1.0,Commercial & Professional Services,E M L CONSULTANTS PLC
3,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,NDB,1.678732e+09,1.0,Banks,NATIONAL DEVELOPMENT BANK PLC
4,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,SAMP,1.678300e+09,1.0,Banks,SAMPATH BANK PLC
...,...,...,...,...,...,...,...
53071,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,PABC,1.678127e+09,4.0,Banks,PAN ASIA BANKING CORPORATION PLC
53072,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,CIC,1.667500e+09,1.0,Materials,C I C HOLDINGS PLC
53073,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,HHL,1.667155e+09,2.0,Capital Goods,HEMAS HOLDINGS PLC
53074,"[COMB, PLC, MGT, ALUM, SUN, ACL, CALT, JAT, AE...",SCB-1522-LC/00,KHL,1.662662e+09,1.0,Consumer Services,JOHN KEELLS HOTELS PLC


In [50]:
test_hoo

,USER_ID,CDSACCNO,STOCKCODE,UNIX_TS,RATING,GICS,STOCKNAME
0,"[SLTL, RIL, EML, NDB, SAMP, EMER, COCO, BALA, ...",BMS-10544-LC/00,VONE,1.641148e+09,3.0,Utilities,VALLIBEL ONE PLC
1,"[SUN, SAMP, RIL, VONE, COMB, DFCC, LMF, TAP, G...",BMS-11214-LC/00,KZOO,1.641148e+09,3.0,Diversified Financials,SHAW WALLACE INVESTMENTS PLC
2,"[PLC, PABC, TILE, SEYB, MGT, TKYO, ACL, TJL, V...",BMS-11807-LC/00,LIOC,1.681670e+09,5.0,Energy,LANKA IOC PLC
3,"[LPL, TILE, CCS, COMB, CFVF, LWL, RICH, SCAP, ...",BMS-11829-LI/00,BERU,1.699900e+09,2.0,Consumer Services,BERUWALA RESORTS PLC
4,"[EDEN, PLR, TKYO, VONE, EXPO, DIAL, SEMB, LIOC...",BMS-12282-LI/00,AMF,1.642444e+09,2.0,Diversified Financials,ASSOCIATED MOTOR FINANCE COMPANY PLC
...,...,...,...,...,...,...,...
2791,"[MBSL, MEL, TESS, LUMX, AGAL, PABC, EXPO, HNB,...",RPS-953190630-VN/00,ALHP,1.647887e+09,1.0,Consumer Services,ANILANA HOTELS AND PROPERTIES PLC
2792,"[DOCK, CDB, HEXP, CARE, LFIN, NDB, PLR, TSML, ...",SBK-80957-LC/00,RCL,1.687459e+09,1.0,Capital Goods,ROYAL CERAMICS LANKA PLC
2793,"[GLAS, PLC, COMB, MGT, ALUM, SUN, ACL, CALT, J...",SCB-11576-LC/00,TJL,1.646937e+09,2.0,Consumer Durables & Apparel,TEEJAY LANKA PLC
2794,"[LLUB, GLAS, DIST, COMB, MGT, PLC, ALUM, SUN, ...",SCB-11577-LC/00,TJL,1.646937e+09,2.0,Consumer Durables & Apparel,TEEJAY LANKA PLC


In [51]:
data_dict = train_hoo.to_dict(orient='list')
train_hoo_dataset = tf.data.Dataset.from_tensor_slices(data_dict)
train_hoo_dataset.save("../../data/portfolios_v2_fixed_port_size_{}_useridseq/retriver_hoo_train".format(port_size))

In [52]:
data_dict = test_hoo.to_dict(orient='list')
test_hoo_dataset = tf.data.Dataset.from_tensor_slices(data_dict)
test_hoo_dataset.save("../../data/portfolios_v2_fixed_port_size_{}_useridseq/retriver_hoo_test".format(port_size))

In [53]:
len(dataset), len(train_hoo_dataset), len(test_hoo_dataset)

(55872, 53076, 2796)

In [54]:
# tf.random.set_seed(42)
# shuffled = dataset.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

# train = shuffled.take(int(len(dataset)* 0.8))
# test = shuffled.skip(int(len(dataset)* 0.8)).take(int(len(dataset)* 0.2))

In [55]:
# train.save("../../data/portfolios_v2/retriver_train")
# test.save("../../data/portfolios_v2/retriver_test")

In [56]:
# new_dataset = tf.data.Dataset.load("../../data/portfolios_v2/portfolios_tfds")

In [57]:
sys.exit()

SystemExit: 

c:\Users\bpadmin\anaconda3\envs\atrad_cars_v2\lib\site-packages\IPython\core\interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Work Here

In [ ]:
import array
import collections

from typing import Dict, List, Optional, Text, Tuple

def _create_feature_dict() -> Dict[Text, List[tf.Tensor]]:
  return {"STOCKCODE": [], "RATING": [], "GICS": [], "STOCKNAME": [], "UNIX_TS": []}

def _sample_list(
    feature_lists: Dict[Text, List[tf.Tensor]],
    num_examples_per_list: int,
    random_state: Optional[np.random.RandomState] = None,
) -> Tuple[tf.Tensor, tf.Tensor]:
  """Function for sampling a list example from given feature lists."""
  if random_state is None:
    random_state = np.random.RandomState()

  sampled_indices = random_state.choice(
      range(len(feature_lists["STOCKCODE"])),
      size=num_examples_per_list,
      replace=False,
  )
  sampled_STOCKCODE = [
      feature_lists["STOCKCODE"][idx] for idx in sampled_indices
  ]
  sampled_RATING = [
      feature_lists["RATING"][idx]
      for idx in sampled_indices
  ]
  sampled_GICS = [
      feature_lists["GICS"][idx] for idx in sampled_indices
  ]
  sampled_STOCKNAME = [
      feature_lists["STOCKNAME"][idx]
      for idx in sampled_indices
  ]
  sampled_UNIX_TS = [
      feature_lists["UNIX_TS"][idx] for idx in sampled_indices
  ]

  return (
      tf.stack(sampled_STOCKCODE, 0),
      tf.stack(sampled_RATING, 0),
      tf.stack(sampled_GICS, 0),
      tf.stack(sampled_STOCKNAME, 0),
      tf.stack(sampled_UNIX_TS, 0)
  )


def sample_listwise(
    rating_dataset: tf.data.Dataset,
    num_list_per_user: int = 10,
    num_examples_per_list: int = 10,
    seed: Optional[int] = None,
) -> tf.data.Dataset:
  
  random_state = np.random.RandomState(seed)

  example_lists_by_user = collections.defaultdict(_create_feature_dict)

  movie_title_vocab = set()
  for example in rating_dataset:
    user_id = example["CDSACCNO"].numpy()
    example_lists_by_user[user_id]["STOCKCODE"].append(
        example["STOCKCODE"])
    example_lists_by_user[user_id]["RATING"].append(
        example["RATING"])
    example_lists_by_user[user_id]["GICS"].append(
        example["GICS"])
    example_lists_by_user[user_id]["STOCKNAME"].append(
        example["STOCKNAME"])
    example_lists_by_user[user_id]["UNIX_TS"].append(
        example["UNIX_TS"])
    
    movie_title_vocab.add(example["STOCKNAME"].numpy())

    

  tensor_slices = {"CDSACCNO": [], "STOCKCODE": [], "RATING": [], "GICS": [], "STOCKNAME": [], "UNIX_TS": []}

  for user_id, feature_lists in example_lists_by_user.items():
    for _ in range(num_list_per_user):

      # Drop the user if they don't have enough ratings.
      if len(feature_lists["STOCKNAME"]) < num_examples_per_list:
        continue

        '''sampled_STOCKCODE, 0),
      tf.stack(sampled_RATING, 0),
      tf.stack(sampled_GICS, 0),
      tf.stack(sampled_STOCKNAME, 0),
      tf.stack(sampled_UNIX_TS'''

      sampled_STOCKCODE, sampled_RATING, sampled_GICS, sampled_STOCKNAME, sampled_UNIX_TS  = _sample_list(
          feature_lists,
          num_examples_per_list,
          random_state=random_state,
      )
      tensor_slices["CDSACCNO"].append(user_id)
      tensor_slices["STOCKCODE"].append(sampled_STOCKCODE)
      tensor_slices["RATING"].append(sampled_RATING)
      tensor_slices["GICS"].append(sampled_GICS)
      tensor_slices["STOCKNAME"].append(sampled_STOCKNAME)
      tensor_slices["UNIX_TS"].append(sampled_UNIX_TS)

  return tf.data.Dataset.from_tensor_slices(tensor_slices)

In [ ]:
# portfolios = tf.data.Dataset.load("../../data/portfolios_tfds_lists")
portfolios = dataset

In [ ]:
# train_ds = tf.data.Dataset.load("D:/dev work/recommender systems/Atrad_CARS/data/train_lists").cache() #data\ratings_train
# test_ds = tf.data.Dataset.load("D:/dev work/recommender systems/Atrad_CARS/data/test_lists").cache()

train_ds = train
test_ds = test

NameError: name 'train' is not defined

In [ ]:
next(iter(train_ds)), len(train_ds)

({'CDSACCNO': <tf.Tensor: shape=(), dtype=string, numpy=b'HDF-733381418-VN/00'>,
  'STOCKCODE': <tf.Tensor: shape=(), dtype=string, numpy=b'LWL'>,
  'UNIX_TS': <tf.Tensor: shape=(), dtype=float32, numpy=1660501800.0>,
  'RATING': <tf.Tensor: shape=(), dtype=float32, numpy=2.0>,
  'GICS': <tf.Tensor: shape=(), dtype=string, numpy=b'Capital Goods'>,
  'STOCKNAME': <tf.Tensor: shape=(), dtype=string, numpy=b'LANKA WALLTILE PLC'>},
 126283)

In [ ]:
train_v1 = sample_listwise(
    train_ds,
    num_list_per_user=50,
    num_examples_per_list=10,
    seed=42
)

test_v1 = sample_listwise(
    test_ds,
    num_list_per_user=1,
    num_examples_per_list=10,
    seed=42
)

In [ ]:
next(iter(train_v1))

{'CDSACCNO': <tf.Tensor: shape=(), dtype=string, numpy=b'HDF-733381418-VN/00'>,
 'STOCKCODE': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'LIOC', b'LITE', b'MBSL', b'PLC', b'LWL', b'ALLI', b'SHL',
        b'HAYL', b'CFVF', b'KAHA'], dtype=object)>,
 'RATING': <tf.Tensor: shape=(10,), dtype=float32, numpy=array([2., 1., 1., 1., 2., 2., 1., 1., 1., 2.], dtype=float32)>,
 'GICS': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'Energy', b'Capital Goods', b'Diversified Financials',
        b'Diversified Financials', b'Capital Goods',
        b'Diversified Financials', b'Capital Goods', b'Capital Goods',
        b'Diversified Financials', b'Food Beverage & Tobacco'],
       dtype=object)>,
 'STOCKNAME': <tf.Tensor: shape=(10,), dtype=string, numpy=
 array([b'LANKA IOC PLC', b'LAXAPANA BATTERIES PLC',
        b'MERCHANT BANK OF SRI LANKA & FINANCE PLC',
        b"PEOPLE'S LEASING & FINANCE PLC", b'LANKA WALLTILE PLC',
        b'ALLIANCE FINANCE COMPANY PLC', b'SOFTLOGIC HOL

In [ ]:
len(train_v1)

258200

In [ ]:
train_v1.save("../../data/portfolios_v2/ranker_train")
test_v1.save("../../data/portfolios_v2/ranker_test")

In [ ]:
train_ds = tf.data.Dataset.load("D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2\retriver_train").cache()

InvalidArgumentError: NewRandomAccessFile failed to Create/Open: D:\dev workecommender systems\Atrad_CARS\data\portfolios_v2etriver_train\dataset_spec.pb : The filename, directory name, or volume label syntax is incorrect.
; no protocol option